# 02 — Vector & Hybrid Retrieval Experiments
**Owner:** Mariem (`feat/retrieval`)  
**Stack:** `BAAI/bge-m3` (Dense Embeddings) + `FAISS` + Reciprocal Rank Fusion (RRF) + `BAAI/bge-reranker-base` (Cross-Encoder)

This notebook validates the retrieval pipeline for DevDoc Copilot:
1. Inspects the pre-built FAISS index (`data/faiss.index`) and chunk metadata (`data/faiss_meta.jsonl`).
2. Benchmarks dense vector search (`search_faiss`) latency and top-$k$ relevance.
3. Compares initial FAISS ranking vs. `BAAI/bge-reranker-base` cross-encoder reranking across multi-framework technical queries.
4. Verifies Reciprocal Rank Fusion (`rrf_merge`) and the `knowledge_retriever.run()` LangGraph node.

In [ ]:
import json
import sys
import time
from collections import Counter
from pathlib import Path

import faiss

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from retrieval.faiss_retriever import search_faiss
from retrieval.hybrid_retriever import retrieve, rrf_merge
from generation.reranker import rerank
from agents.knowledge_retriever import run as run_knowledge_retriever

print(f'Project root: {PROJECT_ROOT}')

## 1. FAISS Index & Corpus Metadata Inspection

In [ ]:
index_path = PROJECT_ROOT / 'data' / 'faiss.index'
meta_path = PROJECT_ROOT / 'data' / 'faiss_meta.jsonl'

index = faiss.read_index(str(index_path))
with open(meta_path, 'r', encoding='utf-8') as f:
    metadata = [json.loads(line) for line in f if line.strip()]

framework_counts = Counter(row.get('framework', 'unknown') for row in metadata)

print(f'FAISS total vectors : {index.ntotal}')
print(f'Embedding dimension : {index.d}')
print(f'Metadata records    : {len(metadata)}')
print(f'Index file size     : {index_path.stat().st_size / (1024 * 1024):.2f} MB')
print('\nChunks per framework:')
for fw, count in framework_counts.most_common():
    print(f'  - {fw:<15}: {count}')

## 2. Dense Retrieval vs. Cross-Encoder Reranking Benchmark

We over-fetch `k=15` candidates from FAISS using `BAAI/bge-m3` and re-score every `(query, chunk)` pair jointly with `BAAI/bge-reranker-base` to promote exact technical matches into the final `top_k=5`.

In [ ]:
benchmark_queries = [
    'How do I define conditional edges and state in LangGraph?',
    'How does FastAPI validate request bodies using Pydantic models?',
    'How do I configure multi-service networking and healthchecks in Docker Compose?',
    'How do I prevent overfitting with early stopping and dropout in PyTorch?',
]

for query in benchmark_queries:
    t0 = time.perf_counter()
    candidates = search_faiss(query, k=15)
    t_faiss = (time.perf_counter() - t0) * 1000

    t1 = time.perf_counter()
    reranked = rerank(query=query, chunks=candidates, top_k=5, enabled=True)
    t_rerank = (time.perf_counter() - t1) * 1000

    print('=' * 80)
    print(f'QUERY: {query}')
    print(f'Latency -> FAISS (k=15): {t_faiss:.1f} ms | Reranker (top_k=5): {t_rerank:.1f} ms')
    print('\nBefore Reranking (Top 3 FAISS):')
    for rank, c in enumerate(candidates[:3], 1):
        print(f'  {rank}. [{c["score"]:.4f}] ({c["framework"]}) {c["chunk_id"]} — {c["text"][:90].replace(chr(10), " ")}...')

    print('\nAfter Reranking (Top 3 Cross-Encoder):')
    for rank, c in enumerate(reranked[:3], 1):
        print(f'  {rank}. [rerank={c["rerank_score"]:.4f} | faiss={c["score"]:.4f}] ({c["framework"]}) {c["chunk_id"]} — {c["text"][:90].replace(chr(10), " ")}...')

## 3. Reciprocal Rank Fusion (RRF) & End-to-End `knowledge_retriever` Node Validation

In [ ]:
# Demonstrate RRF boosting a chunk retrieved by both Dense Search and Graph Search
dense_sample = search_faiss('How does StateGraph work in LangGraph?', k=5)
simulated_graph_hits = [
    dense_sample[2],  # Promote rank-3 dense hit to rank-1 in graph search
    dense_sample[0],
]

fused = rrf_merge([dense_sample, simulated_graph_hits], k_rrf=60, top_k=5)
print('Top 3 after RRF Fusion:')
for i, item in enumerate(fused[:3], 1):
    print(f'  {i}. {item["chunk_id"]} | rrf_score={item["rrf_score"]:.5f}')

# End-to-end LangGraph node execution
sample_state = {
    'question': 'How do I compile a StateGraph in LangGraph?',
    'top_k': 3,
    'use_graph': True,
    'use_reranker': True,
}
node_output = run_knowledge_retriever(sample_state)
print('\nNode output keys:', list(node_output.keys()))
print('Returned chunks :', len(node_output['chunks']))
print('Graph payload   :', node_output['graph'])